# Domain-Specific RAG Chatbot using ChromaDB

## Retrieval-Augmented Generation with Google's Gemini

### Objective

The objective of this project is to build a domain-specific chatbot capable of answering questions from a custom PDF knowledge base.

Instead of relying only on the knowledge already present in a Large Language Model (LLM), Retrieval-Augmented Generation (RAG) retrieves relevant information from a document collection before generating a response.

This project uses the book **Python for Everybody** as the knowledge source.

---

## Workflow

1. Load PDF
2. Extract Text
3. Split into Chunks
4. Generate Embeddings
5. Store in ChromaDB
6. Retrieve Relevant Chunks
7. Generate Response using Gemini
8. Deploy using Streamlit

---

## Technologies Used

- Python
- Streamlit
- Google Gemini
- ChromaDB
- Sentence Transformers
- LangChain
- PyPDF

---

## Expected Outcome

A chatbot capable of answering questions related to Python programming using information retrieved directly from the uploaded PDF.

# Import Required Libraries

The following libraries are required for PDF loading, text splitting, embedding generation, vector database storage, retrieval, and interaction with Google's Gemini model.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [1]:
import os

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from langchain_chroma import Chroma

import google.generativeai as genai

C:\Users\THINK BOOK\AppData\Local\Temp\ipykernel_26464\758590205.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Load the PDF Document

The first step in a Retrieval-Augmented Generation (RAG) system is loading the knowledge source.

For this project, the book **Python for Everybody** is used as the domain-specific corpus. The PDF is loaded using PyPDF, and the text from every page is extracted for further processing.

In [2]:
pdf_path = "data/pythonlearn.pdf"

reader = PdfReader(pdf_path)

print("Total Pages :", len(reader.pages))

Total Pages : 241


# Extract Text from the PDF

Each page of the PDF is processed individually.

The extracted text is combined into a single string, which will later be divided into smaller chunks for semantic search.

In [3]:
text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted + "\n"

In [4]:
print("Total Characters :", len(text))

Total Characters : 430998


# Preview the Extracted Text

Before creating embeddings, it is important to verify that the PDF has been read correctly.

In [11]:
print(text[:2000])

Python for Everybody
Exploring Data Using Python 3
Dr. Charles R. Severance
Credits
Editorial Support: Elliott Hauser, Sue Blumenberg
Cover Design: Aimee Andrion
Printing History
• 2024-Jan-01 Update examples to Python 3.12, remove references to Twitter
APIs, rewrite Databases chapter
• 2023-Jun-29 Many errata included, switch from Google APIs to Open-
StreetMap APIs
• 2016-Jul-05 First Complete Python 3.0 version
• 2015-Dec-20 Initial Python 3.0 rough conversion
Copyright Details
Copyright 2009- Dr. Charles R. Severance.
This work is licensed under a Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 Unported License. This license is available at
http://creativecommons.org/licenses/by-nc-sa/3.0/
You can see what the author considers commercial and non-commercial uses of this
material as well as license exemptions in the Appendix titled “Copyright Detai l” .
iii
Preface
Remixing an Open Book
It is quite natural for academics who are continuously told to “publish or peri sh”
to 

# Text Chunking

Large Language Models cannot efficiently process an entire book at once.

Therefore, the extracted text is divided into smaller overlapping chunks.

Chunking improves retrieval quality by allowing the system to search only the most relevant sections of the document.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

In [6]:
print("Total Chunks :", len(chunks))

Total Chunks : 715


In [14]:
print(chunks[0])

Python for Everybody
Exploring Data Using Python 3
Dr. Charles R. Severance
Credits
Editorial Support: Elliott Hauser, Sue Blumenberg
Cover Design: Aimee Andrion
Printing History
• 2024-Jan-01 Update examples to Python 3.12, remove references to Twitter
APIs, rewrite Databases chapter
• 2023-Jun-29 Many errata included, switch from Google APIs to Open-
StreetMap APIs
• 2016-Jul-05 First Complete Python 3.0 version
• 2015-Dec-20 Initial Python 3.0 rough conversion
Copyright Details
Copyright 2009- Dr. Charles R. Severance.
This work is licensed under a Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 Unported License. This license is available at
http://creativecommons.org/licenses/by-nc-sa/3.0/
You can see what the author considers commercial and non-commercial uses of this


# Generate Text Embeddings

Large Language Models cannot search plain text efficiently.

Each text chunk is converted into a numerical representation called an **embedding** using the Sentence Transformer model.

Chunks with similar meanings produce similar embeddings, enabling semantic search.

In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


# Create the Vector Database

The generated embeddings are stored in ChromaDB.

Instead of searching the entire document, ChromaDB retrieves only the most relevant chunks for each user query, making the chatbot efficient and accurate.

In [8]:
from langchain_core.documents import Document

In [10]:
documents = []

for chunk in chunks:
    documents.append(
        Document(page_content=chunk)
    )

print("Documents Created :", len(documents))

Documents Created : 715


In [11]:
from langchain_core.embeddings import Embeddings

In [12]:
class SentenceTransformerEmbeddings(Embeddings):

    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

In [13]:
embedding_function = SentenceTransformerEmbeddings(
    embedding_model
)
print("Embedding Function Created Successfully.")

Embedding Function Created Successfully.


# Store Embeddings in ChromaDB

Each document chunk is embedded and stored inside a persistent Chroma vector database.

This database will later be queried to retrieve the most relevant knowledge for answering user questions.

In [14]:
vector_db = Chroma.from_documents(

    documents=documents,

    embedding=embedding_function,

    persist_directory="chroma_db"

)
print("Vector Database Created Successfully.")

Vector Database Created Successfully.


# Test Semantic Retrieval

Before integrating the Large Language Model, it is important to verify that the vector database retrieves relevant document chunks for a given query.

In [15]:
query = "What is a Python dictionary?"

results = vector_db.similarity_search(
    query,
    k=3
)

In [16]:
for i, doc in enumerate(results, 1):

    print("=" * 80)

    print(f"Result {i}\n")

    print(doc.page_content[:1000])

Result 1

8.16. EXERCISES 107
Enter a number: 6
Enter a number: 2
Enter a number: 9
Enter a number: 3
Enter a number: 5
Enter a number: done
Maximum: 9.0
Minimum: 2.0
108 CHAPTER 8. LISTS
Chapter 9
Dictionaries
A dictionary is like a list, but more general. In a list, the index positions have to
be integers; in a dictionary, the indices can be (almost) any type.
You can think of a dictionary as a mapping between a set of indices (which are
called keys) and a set of values. Each key maps to a value. The association of a
key and a value is called a key-value pair or sometimes an item.
As an example, we’ll build a dictionary that maps from English to Spanish words,
so the keys and the values are all strings.
The function dict creates a new dictionary with no items. Because dict is the
Result 2

8.16. EXERCISES 107
Enter a number: 6
Enter a number: 2
Enter a number: 9
Enter a number: 3
Enter a number: 5
Enter a number: done
Maximum: 9.0
Minimum: 2.0
108 CHAPTER 8. LISTS
Chapter 9
Dictionar

# Configure Google's Gemini API

The retrieved document chunks are provided as context to Google's Gemini model.

Instead of answering from general knowledge, Gemini generates answers using only the retrieved information from the PDF.

In [17]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini Connected Successfully.")

Gemini Connected Successfully.


# Retrieval-Augmented Generation Pipeline

The chatbot first retrieves the most relevant document chunks from ChromaDB.

These retrieved passages are then supplied to Gemini as contextual information before generating the final response.

In [18]:
def ask_pdf(question):

    docs = vector_db.similarity_search(
        question,
        k=5
    )

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the information provided below.

If the answer is not available in the provided context,
reply:

"I could not find this information in the uploaded PDF."

Context:
{context}

Question:
{question}

Answer:
"""

    response = model.generate_content(prompt)

    return response.text

# Testing the Chatbot

The following examples demonstrate the chatbot retrieving relevant information from the PDF before generating an answer.

In [19]:
question = "What is a Python dictionary?"

print(ask_pdf(question))

A dictionary is like a list, but more general. In a list, the index positions have to be integers; in a dictionary, the indices can be (almost) any type. You can think of a dictionary as a mapping between a set of indices (which are called keys) and a set of values. Each key maps to a value. The association of a key and a value is called a key-value pair or sometimes an item.


In [20]:
print(ask_pdf("Explain loops"))

A type of flow is called a loop because it loops back around to the top. Each time the body of the loop is executed is called an iteration. The body of the loop should change the value of one or more variables so that the condition eventually becomes false and the loop terminates. The variable that changes each time the loop executes and controls when the loop finishes is called the iteration variable.


In [21]:
print(ask_pdf("What is object oriented programming?"))

In a way, object oriented programming is a way to arrange your code so that you can zoom into 50 lines of the code and understand it while ignoring the other 999,950 lines of code for the moment.


In [22]:
print(ask_pdf("Explain functions in Python"))

Function definitions do not alter the flow of execution of the program. Statements inside a function are not executed until the function is called. A function call acts like a detour: the flow of execution jumps to the function's body, executes all statements there, and then returns to where it left off. Functions can also call other functions. Python is capable of keeping track of its location during these nested calls.

Functions offer several benefits:
*   They can make a program smaller by eliminating repetitive code, allowing changes to be made in only one place.
*   Dividing a long program into functions enables debugging parts individually before assembling them.
*   Well-designed functions can be reused across multiple programs.
*   Functions can capture and represent specific ideas or operations, such as "find the smallest value in a list of values."


# Demo Questions

In [23]:
demo_questions = [
    "Explain Python dictionaries.",
    "What is Object-Oriented Programming?",
    "What is Quantum Physics?"
]

for q in demo_questions:

    print("="*70)
    print("Question :", q)
    print()

    print(ask_pdf(q))
    print("\n")

Question : Explain Python dictionaries.

A dictionary is like a list, but more general, as its index positions can be almost any type, not just integers. You can think of a dictionary as a mapping between a set of indices, called keys, and a set of values. Each key maps to a value. The association of a key and a value is called a key-value pair or sometimes an item. The function `dict` creates a new dictionary with no items.


Question : What is Object-Oriented Programming?

In a way, object-oriented programming is a way to arrange your code so that you can zoom into 50 lines of the code and understand it while ignoring the other 999,950 lines of code for the moment.


Question : What is Quantum Physics?

I could not find this information in the uploaded PDF.




# Observation

The chatbot successfully answered domain-specific questions by retrieving relevant document chunks from the vector database before generating responses using Gemini.

For questions outside the document's scope, the chatbot appropriately refused to answer, demonstrating the effectiveness of Retrieval-Augmented Generation (RAG) in reducing hallucinations and ensuring grounded responses.

Overall, the project satisfies the core objectives of building a domain-specific AI assistant using semantic search, embeddings, vector databases, and Large Language Models.

# Conclusion

In this project, a Domain-Specific Retrieval-Augmented Generation (RAG) chatbot was successfully developed using a custom document corpus.

The system extracts text from a PDF document, splits it into meaningful chunks, generates semantic embeddings using Sentence Transformers, and stores them in a Chroma vector database for efficient similarity search.

When a user asks a question, the chatbot retrieves the most relevant document chunks and supplies them as context to Google's Gemini model, ensuring that responses are grounded in the uploaded document rather than relying solely on the model's pre-trained knowledge.

The chatbot successfully answers domain-specific questions, provides retrieved context for transparency, and refuses to answer questions that are outside the document's scope.

This project demonstrates the practical application of Retrieval-Augmented Generation (RAG), semantic search, vector databases, embeddings, and Large Language Models for building reliable AI assistants over custom knowledge bases.